In [0]:
SHOW CATALOGS;

In [0]:
-- 1. Switch context to your existing catalog
USE CATALOG catalog_dev;

-- 2. Create target schema inside catalog_dev
CREATE SCHEMA IF NOT EXISTS schema_migration
COMMENT 'Target schema for relational database migration to Delta Lake';

-- 3. Set the active schema context
USE SCHEMA schema_migration;

-- 4. Verify your active namespace context
SELECT current_catalog(), current_schema();

In [0]:
%python
from pyspark.sql.functions import current_timestamp, col, trim
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

# 1. Create Synthetic Customer Data
customers_data = [
    (1, "Alice Smith", "alice.smith@example.com", "555-0192", "New York"),
    (2, "Bob Jones", "bob.jones@example.com", "555-0193", "Chicago"),
    (3, "Charlie Brown", "charlie.b@example.com", "555-0194", "Houston"),
    (4, "Diana Prince", "diana.p@example.com", "555-0195", "Seattle")
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True)
])

df_customers = spark.createDataFrame(customers_data, customers_schema) \
    .withColumn("_ingested_at", current_timestamp())

df_customers.write.format("delta").mode("overwrite").saveAsTable("catalog_dev.schema_migration.dbo_Customers")

# 2. Create Synthetic Orders Data
orders_data = [
    (101, 1, "2026-01-15 10:30:00", 250.50, "COMPLETED"),
    (102, 2, "2026-01-16 11:45:00", 120.00, "COMPLETED"),
    (103, 1, "2026-01-17 14:20:00", 450.00, "PENDING"),
    (104, 3, "2026-01-18 09:15:00", 85.25, "CANCELLED")
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("status", StringType(), True)
])

df_orders = spark.createDataFrame(orders_data, orders_schema) \
    .withColumn("order_date", col("order_date").cast(TimestampType())) \
    .withColumn("_ingested_at", current_timestamp())

df_orders.write.format("delta").mode("overwrite").saveAsTable("catalog_dev.schema_migration.dbo_Orders")

# 3. Create Synthetic Products & OrderItems Data
products_data = [(1, "Laptop", "Electronics", 1200.00), (2, "Headphones", "Electronics", 150.00)]
df_products = spark.createDataFrame(products_data, ["product_id", "product_name", "category", "price"]).withColumn("_ingested_at", current_timestamp())
df_products.write.format("delta").mode("overwrite").saveAsTable("catalog_dev.schema_migration.dbo_Products")

items_data = [(1, 101, 1, 1, 1200.00), (2, 102, 2, 1, 150.00)]
df_items = spark.createDataFrame(items_data, ["item_id", "order_id", "product_id", "quantity", "unit_price"]).withColumn("_ingested_at", current_timestamp())
df_items.write.format("delta").mode("overwrite").saveAsTable("catalog_dev.schema_migration.dbo_OrderItems")

print("All source tables successfully written to catalog_dev.schema_migration!")

In [0]:
-- 1. Compare record counts across target tables
SELECT 'dbo_Customers' AS table_name, COUNT(1) AS record_count FROM catalog_dev.schema_migration.dbo_Customers
UNION ALL
SELECT 'dbo_Orders', COUNT(1) FROM catalog_dev.schema_migration.dbo_Orders
UNION ALL
SELECT 'dbo_Products', COUNT(1) FROM catalog_dev.schema_migration.dbo_Products
UNION ALL
SELECT 'dbo_OrderItems', COUNT(1) FROM catalog_dev.schema_migration.dbo_OrderItems;

-- 2. Validate key constraints on Customer entity
SELECT 
    COUNT(1) AS total_records,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS null_emails
FROM catalog_dev.schema_migration.dbo_Customers;

In [0]:
USE CATALOG catalog_dev;
USE SCHEMA schema_migration;

-- 1. Create masking function (shows raw email to account owner/admins, masks for all others)
CREATE OR REPLACE FUNCTION email_mask(email STRING)
RETURN CASE 
  WHEN IS_ACCOUNT_GROUP_MEMBER('account users') THEN CONCAT(SUBSTRING(email, 1, 2), '****@', SPLIT_PART(email, '@', 2))
  ELSE email
END;

-- 2. Attach mask to the email column in dbo_Customers
ALTER TABLE catalog_dev.schema_migration.dbo_Customers 
ALTER COLUMN email SET MASK email_mask;

-- 3. Test querying the masked customer data
SELECT customer_id, name, email, phone FROM catalog_dev.schema_migration.dbo_Customers;